In this part of the homework you will play with LoRA fine-tuning.

This notebook is based on [GenAI course](https://academy.nebius.com/generative-ai/) practice session. Credits: [Alex Umnov](https://www.linkedin.com/in/alex-umnov/)

Fist, as in class, we need to install the required libraries:

In [ ]:
!pip install -q peft transformers datasets einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


And then import all the modules:

In [ ]:
import os

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from tqdm import tqdm
import torch

We will use the same dataset as in the class. Again, we will use `huggingface_boilerplate.py` file to preprocess the data, so you'll need to add it into files in the colab.

In [ ]:
from huggingface_boilerplate import prepare_dataloaders

In [ ]:
model_name = "EleutherAI/pythia-1b-deduped"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding_side='left'
)
model = AutoModelForCausalLM.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

train_dataloader, eval_dataloader, dataset = prepare_dataloaders(
    tokenizer,
    dataset_path="tweet_eval",
    dataset_name="irony",
    text_column="text",
    label_names_column="label",
    max_length=64,
    batch_size=8
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

...

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Dataset_sample: {'text': 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life', 'label': 1, 'text_label': 'irony'}


Running tokenizer on dataset:   0%|          | 0/2862 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/784 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/955 [00:00<?, ? examples/s]

And again, lets first see what our base model generates:

In [ ]:
from IPython.display import display

print("Samples")

input_text = [
    f"Tweet text: {text} Label : "
    for text in dataset['test'][:8]['text']
]

display(input_text)

tokenized = tokenizer(input_text, return_tensors='pt', padding=True)
tokenized = {k: v.cuda() for k, v in tokenized.items()}

model = model.cuda()

output = model.generate(
    **tokenized,
    max_new_tokens=10,
    eos_token_id=tokenizer.eos_token_id
)

print("\n\nGenerations\n\n")

display(tokenizer.batch_decode(output, skip_special_tokens=True))

Samples


['Tweet text: @user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to Label : ',
 'Tweet text: Just walked in to #Starbucks and asked for a "tall blonde" Hahahaha #irony Label : ',
 'Tweet text: #NOT GONNA WIN Label : ',
 'Tweet text: @user He is exactly that sort of person. Weirdo! Label : ',
 "Tweet text: So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm Label : ",
 'Tweet text: Corny jokes are my absolute favorite Label : ',
 'Tweet text: People complain about my backround pic and all I feel is like "hey don\'t blame me, Albert E might have spoken those words" #sarcasm #life Label : ',
 'Tweet text: @user @user Darn, my sock joke needs fixing? Label : ']

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.




Generations




['Tweet text: @user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to Label : ###||#TSU #TSU #TS',
 'Tweet text: Just walked in to #Starbucks and asked for a "tall blonde" Hahahaha #irony Label : ###\n\nI\'m not sure if this is',
 'Tweet text: #NOT GONNA WIN Label : \n#NOT GONNA WIN\n\n#',
 'Tweet text: @user He is exactly that sort of person. Weirdo! Label : \n#1\n\nA:\n\nYou',
 "Tweet text: So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm Label : xtc_tweet_text_1\n",
 'Tweet text: Corny jokes are my absolute favorite Label : \n#1: "I\'m a little bit',
 'Tweet text: People complain about my backround pic and all I feel is like "hey don\'t blame me, Albert E might have spoken those words" #sarcasm #life Label : xtian\n\nI\'m not sure if this',
 'Tweet text: @user @user Darn, my sock joke needs fixing? Label : \n#!/usr/bin/env python\n']

As you see, the model generates some random stuff. We need to teach it to respect the format of the answer, and we'll do it through fine-tuning.

## Task 1. LoRA fine-tuning

LoRA doesn't actually change the weights of a model, it rather traines a matrix, which is added to the model's weights. So it doesn't have to work with weights directly and keep their gradients in memory. Furthermore, to reduce memory consumption LoRA works in much smaller dimension, by decomposing this increment matrix into two transformations: to and from lower rank. Hence the name **Low-Rank Adaptaion**.



**Your task will be to play with LoRA hyperparameters and try to achieve best accuracy on test set**. Below there's a code from the class that you can use. You can change both LoRA config parameters (rank should be important) and external parameters, such as learning rate, batch size or number of training epochs.

You might want to add accuracy calculation when evaluating on validation data after each epoch, so you can monitor this metric. It is also beneficial to add visualisation of loss function during training.

As always, try to modify one parameter at time and see how this affects training and testing. Log your experiments so you can remember what you've done and see the whole picture.

Try to answer the following questions:
- What hyperparameters affect the validation/test accuracy the most?
- How do hyperparameters affect training time? Does training time scale linearly with number of training parameters? (you can use [time](https://docs.python.org/3/library/time.html) library to measure execution time in cell)
- Can you observe overfitting? If yes, when does it happen?
- (bonus) What if we change label `'not irony'` into `'neutral'`? I.e. we will have labels `'irony'` and `'neutral'`. Does this change the performance of the model?

In the end, describe your findings.

---------

Let's import all the needed modules from peft and initialise LoRA:

In [ ]:
from peft import (
    get_peft_config,
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftType
)

You can find LoRA config arguments in [Huggingface documentation](https://huggingface.co/docs/peft/en/package_reference/lora). Try to use some that you think might be beneficial.

In [ ]:
# modify config as you like
peft_config = LoraConfig(
    r= 16, # YOUR CODE HERE,
    target_modules=[
        'query_key_value',
        'dense',
        'dense_h_to_4h',
        'dense_4h_to_h'
    ]
)

Initialise the model:

In [ ]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


lr = 1e-5 # YOUR CODE HERE
num_epochs = 10 # YOUR CODE HERE

trainable params: 8,388,608 || all params: 1,020,170,240 || trainable%: 0.8223


As you can see, we are still training just a tiny fraction of the model's parameters.

In [ ]:
# you can change the optimiser
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
# you can change the scheduler and its parameters
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

In [ ]:
model = model.cuda()

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.cuda() for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.cuda() for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(
                torch.argmax(outputs.logits, -1).detach().cpu().numpy(),
                skip_special_tokens=True
            )
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}:\n{train_ppl=}\n{train_epoch_loss=}\n{eval_ppl=}\n{eval_epoch_loss=}")

100%|██████████| 98/98 [00:06<00:00, 14.28it/s]


epoch=0:
train_ppl=tensor(112.5751, device='cuda:0')
train_epoch_loss=tensor(4.7236, device='cuda:0')
eval_ppl=tensor(1.2482, device='cuda:0')
eval_epoch_loss=tensor(0.2217, device='cuda:0')


100%|██████████| 98/98 [00:06<00:00, 14.21it/s]


epoch=1:
train_ppl=tensor(1.2572, device='cuda:0')
train_epoch_loss=tensor(0.2289, device='cuda:0')
eval_ppl=tensor(1.2112, device='cuda:0')
eval_epoch_loss=tensor(0.1916, device='cuda:0')


100%|██████████| 98/98 [00:06<00:00, 14.23it/s]


...

epoch=7:
train_ppl=tensor(1.1673, device='cuda:0')
train_epoch_loss=tensor(0.1547, device='cuda:0')
eval_ppl=tensor(1.2211, device='cuda:0')
eval_epoch_loss=tensor(0.1998, device='cuda:0')


100%|██████████| 98/98 [00:06<00:00, 14.23it/s]


epoch=8:
train_ppl=tensor(1.1606, device='cuda:0')
train_epoch_loss=tensor(0.1489, device='cuda:0')
eval_ppl=tensor(1.2066, device='cuda:0')
eval_epoch_loss=tensor(0.1878, device='cuda:0')


100%|██████████| 98/98 [00:06<00:00, 14.25it/s]

epoch=9:
train_ppl=tensor(1.1566, device='cuda:0')
train_epoch_loss=tensor(0.1455, device='cuda:0')
eval_ppl=tensor(1.2105, device='cuda:0')
eval_epoch_loss=tensor(0.1911, device='cuda:0')


And now it's time to evaluate our model:

In [ ]:
from IPython.display import display

print("Samples")

input_text = [
    f"Tweet text: {text} Label : "
    for text in dataset['test'][:8]['text']
]

display(input_text)

tokenized = tokenizer(input_text, return_tensors='pt', padding=True)
tokenized = {k: v.cuda() for k, v in tokenized.items()}

model = model.cuda()

output = model.generate(
    **tokenized,
    max_new_tokens=10,
    eos_token_id=tokenizer.eos_token_id
)

print("\n\nGenerations\n\n")

display(tokenizer.batch_decode(output, skip_special_tokens=True))

Samples


['Tweet text: @user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to Label : ',
 'Tweet text: Just walked in to #Starbucks and asked for a "tall blonde" Hahahaha #irony Label : ',
 'Tweet text: #NOT GONNA WIN Label : ',
 'Tweet text: @user He is exactly that sort of person. Weirdo! Label : ',
 "Tweet text: So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm Label : ",
 'Tweet text: Corny jokes are my absolute favorite Label : ',
 'Tweet text: People complain about my backround pic and all I feel is like "hey don\'t blame me, Albert E might have spoken those words" #sarcasm #life Label : ',
 'Tweet text: @user @user Darn, my sock joke needs fixing? Label : ']

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.




Generations




['Tweet text: @user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to Label : non irony',
 'Tweet text: Just walked in to #Starbucks and asked for a "tall blonde" Hahahaha #irony Label : irony',
 'Tweet text: #NOT GONNA WIN Label : non irony',
 'Tweet text: @user He is exactly that sort of person. Weirdo! Label : irony',
 "Tweet text: So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm Label : non irony",
 'Tweet text: Corny jokes are my absolute favorite Label : irony',
 'Tweet text: People complain about my backround pic and all I feel is like "hey don\'t blame me, Albert E might have spoken those words" #sarcasm #life Label : irony',
 'Tweet text: @user @user Darn, my sock joke needs fixing? Label : irony']

And here's the code for calculating test accuracy:

In [ ]:
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

eval_texts = [f"Tweet text: {text} Label : "  for text in dataset['test']['text']]
eval_labels = dataset['test']['label']

eval_text_dataloader = DataLoader(
    eval_texts, shuffle=False, batch_size=8
)

model = model.cuda()

output_texts = []

for batch in tqdm(eval_text_dataloader):
    tokenized_batch = tokenizer(
        batch,
        return_tensors='pt',
        padding=True
    )
    tokenized_batch = {
        k: v.cuda() for k, v in tokenized_batch.items()
    }
    output = model.generate(
        **tokenized_batch,
        max_new_tokens=10,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    output_text = tokenizer.batch_decode(
        output,
        skip_special_tokens=True
    )
    output_texts.extend(output_text)

output_labels = [
    1 if "Label : irony" in text else 0
    for text in output_texts
]

accuracy = sum([
    1 if prediction == label else 0
    for label, prediction in zip(eval_labels, output_labels)
]) / len(eval_labels)
accuracy

  0%|          | 0/98 [00:00<?, ?it/s]

0.6785714285714286